In [1]:
import pandas as pd 
import numpy as np 
import parquet

# maps
# import geopandas as gpd
# import json
# import branca.colormap as cm
# import folium
# from folium.plugins import TimestampedGeoJson

# visualization
# import plotly.graph_objects as go
import seaborn as sns 
import matplotlib.pyplot as plt 



## Regressions: 

### Delta trips~ delta stations (panel + first differeciation)
### Poisson trips ~ workday + month + hour
### trip_duration ~ workday + month + hour


In [ ]:
df.head()

,start_station_id,year,start_station_name,start_neighbourhood,start_ward
0,7100,2016,Dundas St E / Regent Park Blvd,Regent Park,Toronto Centre
1,7097,2016,Riverdale Park North (Broadview Ave),North Riverdale,Toronto-Danforth
2,7100,2016,Dundas St E / Regent Park Blvd,Regent Park,Toronto Centre
3,7033,2016,Union Station,St Lawrence-East Bayfront-The Islands,Spadina-Fort York
4,7059,2016,Front St W / Blue Jays Way,Harbourfront-CityPlace,Spadina-Fort York


In [ ]:
df_regr = df[~df.year.isin([2016, 2026])]

In [ ]:
panel = (df_regr
            .groupby(['year', 'start_ward'], as_index=False)
            .agg(stations = ('start_station_id', 'nunique'),
                trips = ('trip_id', 'size'))
            .sort_values(['start_ward', 'year']) 
            
            .assign(stations_lag=lambda d: d.groupby('start_ward')['stations'].shift(1),
                    trips_lag=lambda d: d.groupby('start_ward')['trips'].shift(1))
            
            .assign(
                delta_stations=lambda d: d['stations'] - d['stations_lag'],
                delta_trips=lambda d: d['trips'] - d['trips_lag']
            )
            .dropna(subset=['delta_stations', 'delta_trips'])
 )

In [ ]:
panel

,year,start_ward,stations,trips,stations_lag,trips_lag,delta_stations,delta_trips
10,2018,Beaches-East York,16,12207,7.0,4634,9.0,7573
22,2019,Beaches-East York,32,36629,16.0,12207,16.0,24422
35,2020,Beaches-East York,42,74439,32.0,36629,10.0,37810
54,2021,Beaches-East York,42,81007,42.0,74439,0.0,6568
73,2022,Beaches-East York,49,87621,42.0,81007,7.0,6614
...,...,...,...,...,...,...,...,...
72,2021,York South-Weston,4,4738,4.0,3419,0.0,1319
91,2022,York South-Weston,4,5179,4.0,4738,0.0,441
112,2023,York South-Weston,23,7331,4.0,5179,19.0,2152
135,2024,York South-Weston,27,15241,23.0,7331,4.0,7910


In [ ]:
import statsmodels.api as sm

X = sm.add_constant(panel['delta_stations'])
y = panel['delta_trips']



In [ ]:
model = sm.OLS(y, X).fit()

In [ ]:
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:            delta_trips   R-squared:                       0.070
Model:                            OLS   Adj. R-squared:                  0.063
Method:                 Least Squares   F-statistic:                     10.06
Date:                Thu, 05 Nov 2026   Prob (F-statistic):            0.00187
Time:                        16:15:46   Log-Likelihood:                -1748.5
No. Observations:                 136   AIC:                             3501.
Df Residuals:                     134   BIC:                             3507.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const           2.356e+04   1.08e+04      2.

In [ ]:
model_cl = sm.OLS(y, X).fit(cov_type='cluster', cov_kwds={'groups': panel['start_ward']})
print(model_cl.summary())

                            OLS Regression Results                            
Dep. Variable:            delta_trips   R-squared:                       0.070
Model:                            OLS   Adj. R-squared:                  0.063
Method:                 Least Squares   F-statistic:                     1.797
Date:                Fri, 06 Nov 2026   Prob (F-statistic):              0.194
Time:                        00:23:09   Log-Likelihood:                -1748.5
No. Observations:                 136   AIC:                             3501.
Df Residuals:                     134   BIC:                             3507.
Df Model:                           1                                         
Covariance Type:              cluster                                         
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
const           2.356e+04   1.45e+04      1.

### Get weather data

In [5]:
import holidays

OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"

In [6]:
def fetch_weather(lat, lon, start_date, end_date, timezone="America/Toronto"):
    """Hourly historical weather for one point from Open-Meteo's free archive API
    (no key required). Returns a DataFrame indexed by hourly `date_hour`, columns
    temp_c/precip_mm/snowfall_cm/wind_kph."""
    resp = requests.get(
        OPEN_METEO_URL,
        params={
            "latitude": lat,
            "longitude": lon,
            "start_date": start_date,
            "end_date": end_date,
            "hourly": "temperature_2m,precipitation,snowfall,wind_speed_10m",
            "timezone": timezone,
        },
        timeout=60,
    )
    resp.raise_for_status()
    hourly = resp.json()["hourly"]
    return pd.DataFrame({
        "date_hour": pd.to_datetime(hourly["time"]),
        "temp_c": hourly["temperature_2m"],
        "precip_mm": hourly["precipitation"],
        "snowfall_cm": hourly["snowfall"],
        "wind_kph": hourly["wind_speed_10m"],
    })

In [7]:
def build_holiday_calendar(years):
    """Ontario statutory holidays for the given years as a {date: name} dict,
    for flagging is_holiday on the trips panel."""
    return dict(holidays.Canada(subdiv="ON", years=years))

In [ ]:
weather = fetch_weather(
        lat=43.6532, lon=-79.3832,
        start_date="2016-01-01", end_date=pd.Timestamp.today().strftime("%Y-%m-%d"),
    )

In [8]:
from pathlib import Path
# from utils import fetch_weather, build_holiday_calendar

In [9]:
WEATHER_PATH = Path("data/weather/city=toronto/part.parquet")

In [10]:
import requests

In [11]:
weather = fetch_weather(
        lat=43.6532, lon=-79.3832,
        start_date="2016-01-01", end_date=pd.Timestamp.today().strftime("%Y-%m-%d"),
    )

In [12]:
weather.head()

,date_hour,temp_c,precip_mm,snowfall_cm,wind_kph
0,2016-01-01 00:00:00,-0.0,0.0,0.0,18.9
1,2016-01-01 01:00:00,0.0,0.0,0.0,18.6
2,2016-01-01 02:00:00,0.0,0.0,0.0,20.0
3,2016-01-01 03:00:00,-0.0,0.0,0.0,20.8
4,2016-01-01 04:00:00,-0.1,0.0,0.0,21.3


In [13]:
WEATHER_PATH.parent.mkdir(parents=True, exist_ok=True)

In [14]:
weather.to_parquet(WEATHER_PATH)